# 🍿 CineMatch — Colab Demo

Run the **CineMatch v1 MVP backend** (FastAPI) and its algorithms directly in Google Colab.

This notebook lets you:
1. Install the backend
2. Run the full test suite (38 tests)
3. Demo the core algorithms inline — **no server needed**
4. (Optional) Launch the live API **+ web UI** via an ngrok tunnel

> Built from `CineMatch_PRD_v1`. Colab uses the zero-infra **SQLite** fallback
> (Postgres/Redis/Docker are not available on Colab).

## 1. Get the code

If your repo is **private**, replace the URL with a token URL, e.g.
`https://<TOKEN>@github.com/akhanna222/cinematch.git`.

In [ ]:
import os
REPO_URL = 'https://github.com/akhanna222/cinematch.git'
BRANCH   = 'claude/sweet-shannon-bgo0sb'

if not os.path.isdir('cinematch'):
    !git clone --branch $BRANCH $REPO_URL cinematch
%cd cinematch/backend
print('Now in:', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip -q install -e '.[dev]'
print('\n✅ installed')

## 3. Run the test suite

Validates the algorithm core (compatibility, matching, recommendations, Movie DNA) and the API.

In [ ]:
!python -m pytest -q

## 4. Live algorithm demos (no server)

The `app.core` package is pure Python — import and call it directly.

### 4a. Compatibility score (PRD §7 / §14.3)

In [ ]:
from app.core.movie_dna import MovieDNASnapshot
from app.core.compatibility import compute_compatibility

# Two users with strongly overlapping taste
alex = MovieDNASnapshot(
    genre_weights={'thriller': 0.9, 'sci-fi': 0.7, 'comedy': 0.2},
    positives={'parasite', 'dune', 'se7en', 'fight_club'},
    ratings={'parasite':1.0,'dune':0.6,'se7en':1.0,'fight_club':1.0,'barbie':0.0,'gump':0.6},
    watchlist={'oppenheimer', 'tenet'}, rating_count=24)

sam = MovieDNASnapshot(
    genre_weights={'thriller': 0.8, 'sci-fi': 0.8, 'comedy': 0.3},
    positives={'parasite', 'dune', 'se7en', 'prestige'},
    ratings={'parasite':1.0,'dune':0.6,'se7en':1.0,'fight_club':0.6,'barbie':0.0,'gump':0.6},
    watchlist={'oppenheimer', 'inception'}, rating_count=22)

result = compute_compatibility(alex, sam)
print(f"Compatibility: {result['score']*100:.0f}%\n")
for k, v in result['components'].items():
    print(f'  {k:20s} {v:.3f}')

### 4b. Watch Night matching engine (PRD §14.2)

Aggregates per-participant swipes into consensus picks. Always returns a result.

In [ ]:
from app.core.matching import Swipe, run_match

swipes = [
    Swipe('Alex', 'dune', 'strong_yes'), Swipe('Sam', 'dune', 'interested'),
    Swipe('Alex', 'barbie', 'pass'),     Swipe('Sam', 'barbie', 'interested'),
    Swipe('Alex', 'oppenheimer','interested'), Swipe('Sam','oppenheimer','strong_yes'),
]
for r in run_match(swipes, ['Alex', 'Sam']):
    tag = 'FULL CONSENSUS' if r.full_consensus else 'partial'
    print(f'{r.content_id:12s} score={r.aggregate_score} [{tag}]  not-keen={r.dissenters}')

### 4c. Recommendation levers (PRD §14.4 / §14.5)

The collaborative blend weight `alpha` rises automatically with rating count.

In [ ]:
from app.core.recommendations import collab_alpha, paired_recommend

for n in [0, 25, 50, 100, 500]:
    print(f'{n:4d} ratings -> alpha = {collab_alpha(n):.2f}')

print('\nPaired rec uses min(), not average (PRD AI-02):')
print(paired_recommend({'dune': 3.5, 'tenet': 2.0}, {'dune': 1.2, 'tenet': 2.4}))

## 5. (Optional) Launch the live API + Web UI

Colab has no public port, so we tunnel with **ngrok**.

1. Get a free token: https://dashboard.ngrok.com/get-started/your-authtoken
2. Paste it below and run.

You'll get a public URL — open it to use the **CineMatch web app** (sign up, rate, Watch Night).

In [ ]:
NGROK_AUTH_TOKEN = ''  # <-- paste your ngrok token here

!pip -q install pyngrok nest_asyncio
import nest_asyncio, threading, time, uvicorn
from pyngrok import ngrok
nest_asyncio.apply()

from app.main import app

def _serve():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill()
    url = ngrok.connect(8000).public_url
    print('🍿 CineMatch web app :', url + '/app/')
    print('📖 API docs          :', url + '/docs')
else:
    print('Add your ngrok token above to expose a public URL.')

---
### Notes & limitations on Colab
- Uses **SQLite** (zero infra). Postgres/Redis/Docker need a real host.
- The tunnel + server stop when the Colab runtime idles or resets.
- Recommendation **home feed** needs the TMDB catalogue (next build increment); the
  paired-rec and alpha endpoints work today.

Full architecture & roadmap: see `backend/README.md`.